# AML 3303 — Hands-On: A Mini ETL Pipeline with Quality Checks (30 min)

## Scenario
**ShopNorth**, an online retailer, exports its customer support tickets as a JSON file every day. The
operations team wants them in a database they can query, but the exports are messy: inconsistent
categories, duplicates, broken dates, missing emails, and customers' phone numbers and emails sitting
inside the message text.

You will build the pipeline from the slides, end to end:

```
Extract ─▶ Profile ─▶ Standardise ─▶ Redact PII ─▶ Validate ─┬─▶ Load (tickets) ─▶ Query
                                                             └─▶ Quarantine (bad rows)
                         + a run log every time, + an incremental load on day 2
```

Only `pandas` and Python's built-in `sqlite3` are needed. Nothing to install.

Fill in every cell marked `# TODO`. Hints sit directly above each one.

## 0. Setup

Run once. Do not change it.

In [1]:
# Run this cell once. It creates two daily exports from the ticketing system.
import json, re, time, sqlite3
import pandas as pd

day1 = [
    {"ticket_id": 1001, "customer_email": "priya.s@example.com", "category": "Billing",  "priority": "high",   "created_at": "2026-10-01 09:12", "updated_at": "2026-10-01 09:12", "message": "I was charged twice. Call me at 416-555-0199."},
    {"ticket_id": 1002, "customer_email": "liam.b@example.com",  "category": "shipping", "priority": "Medium", "created_at": "2026-10-01 10:05", "updated_at": "2026-10-01 10:05", "message": "Order hasn't arrived after 10 days."},
    {"ticket_id": 1003, "customer_email": "ana.r@example",       "category": " Returns ", "priority": "low",   "created_at": "2026-10-01 11:40", "updated_at": "2026-10-01 11:40", "message": "How do I return a jacket? Email me at ana.r@example.com"},
    {"ticket_id": 1004, "customer_email": "omar.k@example.com",  "category": "BILLING",  "priority": "urgent", "created_at": "2026-10-01 12:02", "updated_at": "2026-10-01 12:02", "message": "Card ending 4242 declined but money taken."},
    {"ticket_id": 1002, "customer_email": "liam.b@example.com",  "category": "shipping", "priority": "Medium", "created_at": "2026-10-01 10:05", "updated_at": "2026-10-01 10:05", "message": "Order hasn't arrived after 10 days."},
    {"ticket_id": 1005, "customer_email": None,                  "category": "Shipping", "priority": "high",   "created_at": "2026-10-01 14:30", "updated_at": "2026-10-01 14:30", "message": "Package damaged, phone (647) 555-0123."},
    {"ticket_id": 1006, "customer_email": "mei.c@example.com",   "category": "returns",  "priority": "low",    "created_at": "not a date",       "updated_at": "2026-10-01 15:10", "message": "Wrong size sent."},
    {"ticket_id": 1007, "customer_email": "sam.t@example.com",   "category": "Account",  "priority": "medium", "created_at": "2026-10-01 16:45", "updated_at": "2026-10-01 16:45", "message": "Can't reset my password."},
]

day2 = [
    {"ticket_id": 1002, "customer_email": "liam.b@example.com",  "category": "Shipping", "priority": "high",   "created_at": "2026-10-01 10:05", "updated_at": "2026-10-02 08:30", "message": "Still no package. Escalating, call 905-555-0142."},
    {"ticket_id": 1008, "customer_email": "zoe.l@example.com",   "category": "billing",  "priority": "medium", "created_at": "2026-10-02 09:15", "updated_at": "2026-10-02 09:15", "message": "Need an invoice copy for my company."},
    {"ticket_id": 1009, "customer_email": "raj.p@example.com",   "category": "Account",  "priority": "low",    "created_at": "2026-10-02 11:00", "updated_at": "2026-10-02 11:00", "message": "Update my address please."},
    {"ticket_id": 1001, "customer_email": "priya.s@example.com", "category": "Billing",  "priority": "high",   "created_at": "2026-10-01 09:12", "updated_at": "2026-10-01 09:12", "message": "I was charged twice. Call me at 416-555-0199."},
]

json.dump(day1, open("tickets_2026-10-01.json", "w"), indent=1)
json.dump(day2, open("tickets_2026-10-02.json", "w"), indent=1)
print("Exports written: tickets_2026-10-01.json, tickets_2026-10-02.json")

Exports written: tickets_2026-10-01.json, tickets_2026-10-02.json


## 1. Extract (3 min)

Read the day-1 export into a DataFrame.

**Hints:** `pd.read_json("file.json")` or `pd.DataFrame(json.load(open(...)))`

In [2]:
# TODO: load tickets_2026-10-01.json into raw_df and display it
raw_df = pd.read_json("tickets_2026-10-01.json")
raw_df

,ticket_id,customer_email,category,priority,created_at,updated_at,message
0,1001,priya.s@example.com,Billing,high,2026-10-01 09:12,2026-10-01 09:12:00,I was charged twice. Call me at 416-555-0199.
1,1002,liam.b@example.com,shipping,Medium,2026-10-01 10:05,2026-10-01 10:05:00,Order hasn't arrived after 10 days.
2,1003,ana.r@example,Returns,low,2026-10-01 11:40,2026-10-01 11:40:00,How do I return a jacket? Email me at ana.r@ex...
3,1004,omar.k@example.com,BILLING,urgent,2026-10-01 12:02,2026-10-01 12:02:00,Card ending 4242 declined but money taken.
4,1002,liam.b@example.com,shipping,Medium,2026-10-01 10:05,2026-10-01 10:05:00,Order hasn't arrived after 10 days.
5,1005,NaN,Shipping,high,2026-10-01 14:30,2026-10-01 14:30:00,"Package damaged, phone (647) 555-0123."
6,1006,mei.c@example.com,returns,low,not a date,2026-10-01 15:10:00,Wrong size sent.
7,1007,sam.t@example.com,Account,medium,2026-10-01 16:45,2026-10-01 16:45:00,Can't reset my password.


## 2. Profile the data (4 min)

Before changing anything, find out what is wrong. Real exports are messy.

**Hints**
- `df.duplicated().sum()`
- `df.isna().sum()`
- `df["category"].unique()` shows inconsistent spellings
- `pd.to_datetime(df["created_at"], errors="coerce").isna()` finds unparseable dates

In [3]:
# TODO: print duplicates, missing values, the distinct categories and priorities,
#       and how many created_at values cannot be parsed as dates
print("Duplicates rows:", raw_df.duplicated().sum())
print("\nMissing values per column")
print(raw_df.isna().sum())

print("\nCategories:", raw_df['category'].unique())
print("\nPriorities:", raw_df['priority'].unique())

bad_dates = pd.to_datetime(raw_df['created_at'], errors='coerce').isna().sum()
print("\nUnparsable created_at values:", bad_dates)

Duplicates rows: 1

Missing values per column
ticket_id         0
customer_email    1
category          0
priority          0
created_at        0
updated_at        0
message           0
dtype: int64

Categories: <ArrowStringArray>
['Billing', 'shipping', ' Returns ', 'BILLING', 'Shipping', 'returns',
 'Account']
Length: 7, dtype: str

Priorities: <ArrowStringArray>
['high', 'Medium', 'low', 'urgent', 'medium']
Length: 5, dtype: str

Unparsable created_at values: 1


In [4]:
print(raw_df.loc[bad_dates, ['ticket_id', 'created_at']])

ticket_id                 1002
created_at    2026-10-01 10:05
Name: 1, dtype: object


## 3. Transform: standardise (6 min)

Write `standardise(df)` so every later step can rely on clean values.

**Hints**
- Drop exact duplicates: `df.drop_duplicates()`
- Categories: `.str.strip().str.title()` turns `" Returns "` and `"BILLING"` into `"Returns"` and `"Billing"`
- Priority: `.str.lower()`
- Dates: `pd.to_datetime(..., errors="coerce")` turns bad values into `NaT` instead of crashing

In [6]:
def standardise(df):
    df = df.drop_duplicates().copy()
    df["category"] = df["category"].str.strip().str.title()
    df["priority"] = df["priority"].str.strip().str.lower()
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    df["updated_at"] = pd.to_datetime(df["updated_at"], errors="coerce")
    return df

clean_df = standardise(raw_df)
clean_df


,ticket_id,customer_email,category,priority,created_at,updated_at,message
0,1001,priya.s@example.com,Billing,high,2026-10-01 09:12:00,2026-10-01 09:12:00,I was charged twice. Call me at 416-555-0199.
1,1002,liam.b@example.com,Shipping,medium,2026-10-01 10:05:00,2026-10-01 10:05:00,Order hasn't arrived after 10 days.
2,1003,ana.r@example,Returns,low,2026-10-01 11:40:00,2026-10-01 11:40:00,How do I return a jacket? Email me at ana.r@ex...
3,1004,omar.k@example.com,Billing,urgent,2026-10-01 12:02:00,2026-10-01 12:02:00,Card ending 4242 declined but money taken.
5,1005,NaN,Shipping,high,2026-10-01 14:30:00,2026-10-01 14:30:00,"Package damaged, phone (647) 555-0123."
6,1006,mei.c@example.com,Returns,low,NaT,2026-10-01 15:10:00,Wrong size sent.
7,1007,sam.t@example.com,Account,medium,2026-10-01 16:45:00,2026-10-01 16:45:00,Can't reset my password.


## 4. Transform: redact PII (5 min)

Ticket messages contain phone numbers and email addresses. Anything sent onward, to a dashboard or an
LLM, should not carry them. This is the governance step from the slides, and under PIPEDA it is not optional.

**Hints**
- Phone numbers like `416-555-0199` and `(647) 555-0123`: `r"\(?\d{3}\)?[\s-]?\d{3}-\d{4}"`
- Emails: `r"[\w.+-]+@[\w-]+\.[\w.]+"`
- `re.sub(pattern, "[PHONE]", text)`

In [7]:
import re

PHONE = r"\(?\d{3}\)?[\s-]?\d{3}-\d{4}"
EMAIL = r"[\w.+-]+@[\w-]+\.[\w.]+"

def redact(text):
    # replace phones with [PHONE] and emails with [EMAIL]
    if not isinstance(text, str):
        return text
    text = re.sub(EMAIL, "[EMAIL]", text)
    text = re.sub(PHONE, "[PHONE]", text)
    return text

clean_df["message"] = clean_df["message"].apply(redact)
clean_df[["ticket_id", "message"]]

,ticket_id,message
0,1001,I was charged twice. Call me at [PHONE].
1,1002,Order hasn't arrived after 10 days.
2,1003,How do I return a jacket? Email me at [EMAIL]
3,1004,Card ending 4242 declined but money taken.
5,1005,"Package damaged, phone [PHONE]."
6,1006,Wrong size sent.
7,1007,Can't reset my password.


## 5. Validate and quarantine (5 min)

Some rows cannot be fixed automatically. Do not delete them silently and do not load them either.
Move them to a **quarantine** table with the reason, so someone can review them.

Rules for a valid ticket:
1. `customer_email` is present and looks like an email (contains `@` and a `.` after it)
2. `created_at` is a real date
3. `priority` is one of `low`, `medium`, `high`, `urgent`

**Hints**
- `df["customer_email"].str.contains(r"@.+\.", na=False)`
- `df["created_at"].notna()`
- `df["priority"].isin([...])`

In [8]:
VALID_PRIORITIES = ["low", "medium", "high", "urgent"]

def validate(df):
    # build one boolean Series per rule, then combine them
    email_ok = df["customer_email"].str.contains(r"@.+\.", na=False)
    date_ok = df["created_at"].notna()
    priority_ok = df["priority"].isin(VALID_PRIORITIES)
    ok = email_ok & date_ok & priority_ok

    # text column saying which rule(s) failed
    failed = pd.DataFrame({
        "bad customer_email": ~email_ok,
        "bad created_at": ~date_ok,
        "bad priority": ~priority_ok,
    })
    reasons = failed.apply(lambda row: ", ".join(row.index[row]), axis=1)

    return df[ok], df[~ok].assign(reason=reasons[~ok])

good_df, quarantine_df = validate(clean_df)
print(len(good_df), "valid,", len(quarantine_df), "quarantined")
quarantine_df

4 valid, 3 quarantined


,ticket_id,customer_email,category,priority,created_at,updated_at,message,reason
2,1003,ana.r@example,Returns,low,2026-10-01 11:40:00,2026-10-01 11:40:00,How do I return a jacket? Email me at [EMAIL],bad customer_email
5,1005,NaN,Shipping,high,2026-10-01 14:30:00,2026-10-01 14:30:00,"Package damaged, phone [PHONE].",bad customer_email
6,1006,mei.c@example.com,Returns,low,NaT,2026-10-01 15:10:00,Wrong size sent.,bad created_at


## 6. Load into SQLite and log the run (4 min)

SQLite is a full SQL database in a single file, so it stands in for the warehouse.

Every run also writes a row to a `pipeline_runs` table: when it ran, rows in, rows loaded, rows rejected.
That is the monitoring idea from the slides, and the same pattern as logging LLM calls.

**Hints**
- `df.to_sql("tickets", conn, if_exists="replace", index=False)`
- Convert datetimes to text first so SQLite stores them cleanly: `.dt.strftime("%Y-%m-%d %H:%M")`

In [9]:
conn = sqlite3.connect("support.db")

def load(df, table, conn, mode="replace"):
    out = df.copy()
    for col in ["created_at", "updated_at"]:
        out[col] = out[col].dt.strftime("%Y-%m-%d %H:%M")
    # write `out` to `table` using to_sql
    out.to_sql(table, conn, if_exists=mode, index=False)

def log_run(conn, source, rows_in, loaded, rejected, seconds):
    # append one row to a pipeline_runs table
    run = pd.DataFrame([{
        "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "source": source,
        "rows_in": rows_in,
        "loaded": loaded,
        "rejected": rejected,
        "seconds": round(seconds, 3),
    }])
    run.to_sql("pipeline_runs", conn, if_exists="append", index=False)

start = time.time()
load(good_df, "tickets", conn)
load(quarantine_df, "quarantine", conn)
log_run(conn, "tickets_2026-10-01.json", len(raw_df), len(good_df), len(quarantine_df), time.time() - start)

pd.read_sql("SELECT * FROM pipeline_runs", conn)

,run_at,source,rows_in,loaded,rejected,seconds
0,2026-11-22 21:41:52,tickets_2026-10-01.json,8,4,3,0.034


## 7. Incremental load: day 2 (5 min)

Day 2's export has a new ticket, an **updated** ticket (1002 was escalated) and an **old** row that was
already loaded (1001). Reloading everything every day does not scale. Instead, load only what changed.
This is the idea behind CDC.

1. Find the **watermark**: the latest `updated_at` already in the warehouse.
2. Keep only day-2 rows with `updated_at` after the watermark.
3. **Upsert:** delete the old versions of those ticket IDs, then insert the new ones.

**Hints**
- `pd.read_sql("SELECT MAX(updated_at) AS wm FROM tickets", conn)`
- Reuse `standardise`, `redact` and `validate` from above. That is why they are functions.
- `conn.execute(f"DELETE FROM tickets WHERE ticket_id IN ({ids})")`

In [10]:
wm_df = pd.read_sql("SELECT MAX(updated_at) AS wm FROM tickets", conn)
watermark = pd.to_datetime(wm_df["wm"].iloc[0])   # latest updated_at in the tickets table, as a Timestamp
print("watermark:", watermark)

day2_raw = pd.read_json("tickets_2026-10-02.json", dtype={"created_at": str, "updated_at": str})
day2 = standardise(day2_raw)
day2["message"] = day2["message"].apply(redact)

changed = day2[day2["updated_at"] > watermark]   # rows with updated_at after the watermark
print(len(changed), "of", len(day2), "rows changed")

good2, bad2 = validate(changed)
# delete old versions of good2's ticket_ids, then append good2
if len(good2):
    ids = ",".join(str(i) for i in good2["ticket_id"])
    conn.execute(f"DELETE FROM tickets WHERE ticket_id IN ({ids})")
    load(good2, "tickets", conn, mode="append")
    conn.commit()

pd.read_sql("SELECT ticket_id, category, priority, updated_at, message FROM tickets ORDER BY ticket_id", conn)

watermark: 2026-10-01 16:45:00
3 of 4 rows changed


,ticket_id,category,priority,updated_at,message
0,1001,Billing,high,2026-10-01 09:12,I was charged twice. Call me at [PHONE].
1,1002,Shipping,high,2026-10-02 08:30,"Still no package. Escalating, call [PHONE]."
2,1004,Billing,urgent,2026-10-01 12:02,Card ending 4242 declined but money taken.
3,1007,Account,medium,2026-10-01 16:45,Can't reset my password.
4,1008,Billing,medium,2026-10-02 09:15,Need an invoice copy for my company.
5,1009,Account,low,2026-10-02 11:00,Update my address please.


## 8. Use the data (2 min)

The operations team wants open tickets per category, urgent and high first.

**Hint:** `GROUP BY category`, and `SUM(priority IN ('high','urgent'))` counts the high-priority ones.

In [11]:
# TODO: one SQL query: tickets per category, with a count of high/urgent ones
# one SQL query: tickets per category, with a count of high/urgent ones
query = """
SELECT category,
       COUNT(*)                                AS tickets,
       SUM(priority IN ('high', 'urgent'))     AS high_or_urgent
FROM tickets
GROUP BY category
ORDER BY high_or_urgent DESC, tickets DESC
"""
pd.read_sql(query, conn)

,category,tickets,high_or_urgent
0,Billing,3,2
1,Shipping,1,1
2,Account,2,0
